# Lesson 8: LangChain & LangGraph

## Lab 08: Build a LangGraph Agent

So far you've built every agent by hand with the Gemini SDK: tool schemas, loops, retrieval and memory. That's the best way to understand how agents work. Real projects usually use **frameworks** that handle the repetitive parts for you.

In this activity you learn the two most popular ones, **LangChain** and **LangGraph**, by building one project: a **Course Helpdesk agent**. It reads each request, decides what kind it is, and sends it down the right path:

| Request | Path |
|---|---|
| A question about the course | Search the knowledge base, then answer with sources |
| A calculation | Use a calculator tool, looping until done |
| "Send a message to…" | Draft the message, **ask a human to approve**, then send or cancel |
| Greetings and small talk | Reply directly |

It also **remembers** each conversation, so follow-up questions work.

## Learning objectives

By the end of this activity, you should be able to:

1. Explain what LangChain is and when it's useful.
2. Use LangChain models, prompts, tools and retrievers.
3. Explain what LangGraph adds: state, nodes, edges and conditional routing.
4. Build a LangGraph workflow that routes requests to different paths.
5. Add a tool loop, persistence and a human-in-the-loop approval step.

**Time:** about 2 hours
**You'll need:** a Gemini API key, and Lessons 5 and 6 completed (this activity reuses their ideas).

> **How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code below it. Cells marked **Try it** invite you to change something and run again.

## 1. What is LangChain, and why is it useful?

**LangChain** is a Python library of ready-made building blocks for LLM applications. Each block has the **same interface**, so they snap together like LEGO.

Compare it with what you built by hand in earlier lessons:

| You built by hand | LangChain gives you |
|---|---|
| `client.models.generate_content(...)` | A **model** object that works the same for Gemini, OpenAI, Claude and others |
| f-strings for prompts | **Prompt templates** with named slots |
| JSON tool schemas (Week 1) | **Tools** from a Python function and its docstring |
| Embeddings + ChromaDB search (Lesson 5) | **Retrievers** that return the most relevant documents |
| Your own agent loop (Lesson 4) | Prebuilt **agents** |

Every LangChain component has an `.invoke()` method. You'll use it a lot.

### Why learn the hand-built way first?

Frameworks hide the details. Because you've already built tool calling, loops, retrieval and memory yourself, you'll know exactly what each LangChain component is doing behind the scenes, and you'll be able to fix it when it goes wrong.

### The packages

| Package | What it contains |
|---|---|
| `langchain` | Core building blocks and prebuilt agents |
| `langchain-google-genai` | The connection to Gemini models and embeddings |
| `langgraph` | Graph-based workflows: state, nodes, edges, persistence, human-in-the-loop |

### Setup

Install the three packages. This may take a minute.

In [ ]:
!pip -q install langchain langgraph langchain-google-genai

### Add your Gemini API key

Enter your key when prompted. LangChain's Gemini integration reads it from the `GEMINI_API_KEY` environment variable. **Never paste your API key directly into a code cell.**

In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("API key loaded.")

## 2. Models and prompts

### Models

A LangChain **chat model** wraps an LLM in a standard interface. Here it's Gemini, but switching to another provider would only change this one line.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL = "gemini-3.8-flash"
llm = ChatGoogleGenerativeAI(model=MODEL)

reply = llm.invoke("In one sentence, what is LangChain?")
print(reply.text)

`llm.invoke()` returns an **AIMessage**. Its `.text` is the reply.

### Messages

Instead of plain text, you can send a list of **messages**. These are the same roles you used with the Gemini SDK:

| LangChain message | Role |
|---|---|
| `SystemMessage` | Instructions for the model |
| `HumanMessage` | What the user says |
| `AIMessage` | What the model replied |
| `ToolMessage` | The result of a tool call |

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

reply = llm.invoke([
    SystemMessage(content="You are a helpful course assistant. Answer in one short sentence."),
    HumanMessage(content="Why do agents need tools?"),
])
print(reply.text)

### Prompt templates

A **prompt template** is a prompt with named slots, filled in at run time. That keeps your prompt text in one place, separate from your data.

The `|` symbol joins components into a **chain**: the filled-in prompt flows straight into the model.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain AI ideas to beginners in {sentences} sentences, using an everyday example."),
    ("human", "Explain: {topic}"),
])

chain = explain_prompt | llm

print(chain.invoke({"topic": "embeddings", "sentences": 2}).text)

> **Try it:** change the `topic` and the number of `sentences`, then run the cell again.

## 3. Tools and retrievers

### Tools

In Week 1 you wrote a JSON schema for every tool. In LangChain, you write a normal Python function and add the `@tool` decorator. LangChain builds the schema for you from:

- the **function name** (the tool name),
- the **type hints** (the parameters),
- the **docstring** (the description the model reads, so write it carefully).

Here's the safe calculator from Week 1 as a LangChain tool.

In [ ]:
import ast
import operator
from langchain_core.tools import tool

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Only numbers and + - * / ** are allowed")


@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression exactly, for example '0.15 * 2480 + 99'. Use it for any maths."""
    try:
        return str(_safe_eval(ast.parse(expression, mode="eval").body))
    except Exception as e:
        return f"Error: {e}"


TOOLS = [calculate]

print("Name:       ", calculate.name)
print("Description:", calculate.description)
print("Parameters: ", calculate.args)
print("Test:       ", calculate.invoke({"expression": "125 * 24"}))

To let the model use tools, **bind** them to it. The model then decides whether to call a tool, exactly like in Week 1. Nothing is executed yet: we just look at the request.

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)

reply = llm_with_tools.invoke("What is 15% of 2,480?")
print("Tool calls:", reply.tool_calls)

Compare this with Week 1: there you dug through `response.candidates[0].content.parts` to find the function call. Here it's simply `reply.tool_calls`.

### Retrievers

A **retriever** takes a query and returns the most relevant documents. It's the RAG search from Lesson 5 packaged as one component. We'll reuse the **same 8 documents** about Agentic AI.

| Lesson 5 (by hand) | LangChain |
|---|---|
| A dictionary per document | A `Document` with `page_content` and `metadata` |
| `sentence-transformers` embeddings | `GoogleGenerativeAIEmbeddings` (Gemini's embedding model, same API key) |
| ChromaDB collection | `InMemoryVectorStore` |
| Your `knowledge_search()` function | `.as_retriever()` |

In [ ]:
KNOWLEDGE_BASE = [
    {
        "id": "kb-01",
        "title": "What is an AI agent?",
        "text": (
            "An AI agent is a system that uses a language model to pursue a goal by deciding what to do next. "
            "Unlike a chatbot, which only replies to messages, an agent can take actions with tools, look at the "
            "results, and choose its next step. A workflow follows a fixed path written by a developer; an agent "
            "chooses its own path. Most real agents are assisted: a human approves important actions such as "
            "payments or sending emails."
        ),
    },
    {
        "id": "kb-02",
        "title": "Tools and function calling",
        "text": (
            "A tool is a function an agent can ask to use, such as a calculator, a web search or an API. The model "
            "never runs the tool itself. It returns a function call containing the tool name and arguments; your "
            "Python code runs the function and sends the result back. Each tool is described with a schema: a name, "
            "a description and parameters. The model chooses tools using only these descriptions, so clear "
            "descriptions matter more than clever code."
        ),
    },
    {
        "id": "kb-03",
        "title": "The ReAct loop and stopping rules",
        "text": (
            "ReAct stands for Reason and Act. The agent reasons about what it needs, acts by calling a tool, "
            "observes the result, and repeats until it can answer. Every agent loop needs three stopping rules: "
            "stop when the model makes no more tool calls, stop when the step budget runs out, and block repeated "
            "identical calls with a loop guard. Without them, an agent can loop forever and waste money."
        ),
    },
    {
        "id": "kb-04",
        "title": "Agent memory",
        "text": (
            "Short-term memory is the conversation history the agent sends to the model on every turn. It is limited "
            "by the context window, the maximum number of tokens the model can read at once. Long-term memory lives "
            "outside the model, for example in a database, and is retrieved only when needed. Retrieval lets an "
            "agent use far more information than fits in its context window."
        ),
    },
    {
        "id": "kb-05",
        "title": "Retrieval-Augmented Generation (RAG)",
        "text": (
            "RAG has three steps: retrieve relevant documents, augment the prompt with them, and generate an answer "
            "grounded in that text. It reduces hallucination because the model answers from real sources it can "
            "cite. Unlike fine-tuning, RAG does not change the model's weights; to update its knowledge you simply "
            "update the documents. In agentic RAG, retrieval is a tool, and the agent decides when to search and "
            "what to search for."
        ),
    },
    {
        "id": "kb-06",
        "title": "Embeddings and vector databases",
        "text": (
            "An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar "
            "meanings have embeddings that are close together, even if they share no words, so 'car' and 'vehicle' "
            "end up near each other. Closeness is usually measured with cosine similarity. A vector database such "
            "as ChromaDB stores embeddings and quickly finds the ones closest to a query. This is called semantic search."
        ),
    },
    {
        "id": "kb-07",
        "title": "Multi-agent systems",
        "text": (
            "A multi-agent system splits work between several agents with different roles, such as a planner, a "
            "researcher and a writer. An orchestrator agent assigns tasks and combines the results. Agents pass work "
            "to each other through handoffs. Multiple agents help with large or varied tasks, but they cost more and "
            "are harder to debug, so start with a single agent and add more only when you need them."
        ),
    },
    {
        "id": "kb-08",
        "title": "Agent safety and guardrails",
        "text": (
            "Agents act in the real world, so they need guardrails. Give each agent only the tools it needs, a "
            "principle called least privilege. Ask a human to approve risky actions such as payments, deletions or "
            "sending messages. Treat text that comes back from tools and documents as data, never as instructions; "
            "attackers can hide commands in web pages, an attack called prompt injection. Log every step so you can "
            "audit what the agent did."
        ),
    },
]


print(f"{len(KNOWLEDGE_BASE)} documents loaded.")

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings

documents = [
    Document(page_content=doc["text"], metadata={"id": doc["id"], "title": doc["title"]})
    for doc in KNOWLEDGE_BASE
]

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

for doc in retriever.invoke("How do I stop an agent from looping forever?"):
    print("-", doc.metadata["title"])

Three lines replaced most of Lesson 5's setup. The top result should be **The ReAct loop and stopping rules**.

### Agents

LangChain can also build a complete tool-using agent in one line with `create_agent`. It runs the same loop you wrote in Lesson 4: call the model, run any tools it asks for, send back the results, and repeat until there's a final answer.

In [ ]:
from langchain.agents import create_agent

quick_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are a maths helper. Always use the calculate tool for arithmetic.",
)

result = quick_agent.invoke({"messages": [HumanMessage(content="What is 15% of 2,480, plus 99?")]})

for message in result["messages"]:
    print(f"{type(message).__name__:12}", message.tool_calls if getattr(message, "tool_calls", None) else message.text)

You can see the whole loop in the message list: the question, the model's tool call, the tool's result, and the final answer.

`create_agent` is great for a simple "model + tools" agent. But our helpdesk needs more control:

- **different paths** for different kinds of requests,
- a **pause for human approval** before sending anything,
- **memory** of the conversation.

That's what LangGraph is for. (In fact, `create_agent` is itself built with LangGraph.)

## 4. What LangGraph adds to agent workflows

**LangGraph** lets you describe an agent as a **graph**: boxes (steps) connected by arrows (what happens next). You decide the structure, and the LLM makes decisions only where you allow it.

Here's the helpdesk we're building:

```text
                       START
                         ↓
                     classify            ← the LLM decides the route
        ┌───────────┬────┴──────┬──────────────┐
    question    calculation    action          chat
        ↓            ↓            ↓              ↓
    retrieve   calculator_agent ⇄ tools   draft_action    chat
        ↓            ↓                        ↓              ↓
     answer         END                human_approval      END
        ↓                                     ↓          (pauses for a human)
       END                                   END
```

Five ideas make this work:

| Idea | What it is | In our helpdesk |
|---|---|---|
| **State** | A shared notebook that every step reads from and writes to | The messages, the chosen route, retrieved context, the draft message |
| **Nodes** | The steps: plain Python functions | `classify`, `retrieve`, `answer`, `calculator_agent`, ... |
| **Edges** | The arrows: what runs next | `retrieve` → `answer` |
| **Conditional edges** | Arrows that choose a path based on the state | After `classify`, go to one of four paths |
| **Checkpointer** | Saves the state after every step | Remembers conversations and lets the graph pause for approval |

Why not just write a big `if`/`else` in Python? You could, for a small project. A graph also gives you persistence, pausing and resuming, a picture of your workflow, and a clear place for every step, which matters as agents grow.

## 5. State

The **state** is a `TypedDict`: a dictionary where we declare each key and its type up front. Every node receives the full state and returns **only the keys it wants to change**.

One key is special. `messages` uses `Annotated[list, add_messages]`. The `add_messages` part is a **reducer**: it tells LangGraph that when a node returns messages, they should be **added** to the list instead of replacing it. Without it, each node would wipe out the conversation.

In [ ]:
from typing import Annotated, Literal, TypedDict
from langgraph.graph.message import add_messages


class HelpdeskState(TypedDict):
    # `add_messages` means: when a node returns messages, ADD them to the list (don't replace it).
    messages: Annotated[list, add_messages]
    route: str             # "question", "calculation", "action" or "chat"
    context: str           # text retrieved from the knowledge base
    sources: list[str]     # titles of the retrieved documents
    draft: dict            # the message waiting for approval: {"to": ..., "text": ...}
    action_result: str     # what happened to the message: sent or cancelled


# What the reducer does: it adds new messages to the existing list.
print(add_messages([HumanMessage(content="Hi")], [AIMessage(content="Hello!")]))

The other keys have no reducer, so a node's update simply **replaces** the old value. For example, `classify` sets `route`.

## 6. Nodes

A **node** is just a Python function:

```python
def my_node(state):
    ...                          # read what you need from the state
    return {"some_key": value}   # return only the keys you change
```

Our first node, `classify`, decides the route. It uses a LangChain feature called **structured output**: we describe the answer we want as a Pydantic model, and the LLM returns exactly that shape, the same idea as the structured outputs from Week 1. The `Literal` type limits `route` to our four options.

First, the prompts and helper functions the nodes share.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import MessagesPlaceholder


class RouteDecision(BaseModel):
    """The router's decision."""
    route: Literal["question", "calculation", "action", "chat"] = Field(
        description="Which path should handle the user's latest message")
    reason: str = Field(description="One short sentence explaining the choice")


class MessageDraft(BaseModel):
    """A message the agent wants to send."""
    to: str = Field(description="Who the message is for, e.g. 'my study group' or 'the instructor'")
    text: str = Field(description="The message to send, written politely and briefly")


ROUTER_PROMPT = """You route requests for a course helpdesk about Agentic AI. Pick one route:
- question: asks about agentic AI concepts (agents, tools, the ReAct loop, memory, RAG, embeddings,
  multi-agent systems, agent safety)
- calculation: needs arithmetic or any numeric calculation
- action: asks you to send, email, message or remind someone
- chat: greetings, thanks, small talk, or a follow-up about the conversation so far"""

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful course assistant. Answer the user's latest question using only the context below. "
     "If the context doesn't answer it, say the knowledge base doesn't cover it. Keep answers short and clear."
     "\n\nContext:\n{context}"),
    MessagesPlaceholder("messages"),
])

CALCULATOR_PROMPT = ("You are a careful maths assistant. Use the calculate tool for every calculation, "
                     "never mental maths. Then give the final answer in one short sentence.")

CHAT_PROMPT = ("You are a friendly course helpdesk assistant for a beginner Agentic AI course. "
               "You can answer questions about the course topics, do calculations, and send messages "
               "after the user approves them. Keep replies short.")

DRAFT_PROMPT = "Turn the user's request into a short message to send. Don't add details they didn't give."


def last_user_message(state) -> str:
    # The text of the most recent message from the user.
    for message in reversed(state["messages"]):
        if isinstance(message, HumanMessage):
            return message.text
    return ""


def conversation(state) -> list:
    # The conversation so far as plain user/assistant text messages (no tool calls).
    return [m for m in state["messages"]
            if isinstance(m, HumanMessage) or (isinstance(m, AIMessage) and not m.tool_calls and m.text)]


def current_turn(state) -> list:
    # Everything from the latest user message onward, including tool calls and results.
    messages = state["messages"]
    for i in range(len(messages) - 1, -1, -1):
        if isinstance(messages[i], HumanMessage):
            return messages


print("Prompts and helpers ready.")

The three helpers pick the right messages for each node:

| Helper | Returns | Used by |
|---|---|---|
| `last_user_message` | The text of the user's latest message | `classify`, `retrieve`, `draft_action` |
| `conversation` | The chat so far, without tool calls | `answer`, `chat` |
| `current_turn` | This turn only, **including** tool calls and results | `calculator_agent` |

Now the first two nodes.

In [ ]:
def classify(state: HelpdeskState) -> dict:
    # Decide which path should handle the request.
    router = llm.with_structured_output(RouteDecision)
    decision = router.invoke([SystemMessage(content=ROUTER_PROMPT), HumanMessage(content=last_user_message(state))])
    print(f"  [classify] route = {decision.route}  ({decision.reason})")
    return {"route": decision.route}


def chat(state: HelpdeskState) -> dict:
    # Reply directly, using the conversation history.
    reply = llm.invoke([SystemMessage(content=CHAT_PROMPT)] + conversation(state))
    return {"messages": [AIMessage(content=reply.text)]}


# A node is just a function, so we can test it directly with a sample state.
print(classify({"messages": [HumanMessage(content="What is RAG?")]}))
print(classify({"messages": [HumanMessage(content="Good morning!")]}))

Notice that `classify` returns only `{"route": ...}`. It doesn't touch the other keys.

> **Try it:** call `classify` with your own messages. Can you write one that confuses the router?

## 7. Edges

**Edges** connect the nodes. Every graph starts at `START` and finishes at `END`.

Building a graph always takes four steps:

1. Create a `StateGraph` with your state type.
2. `add_node` for each step.
3. `add_edge` to connect them.
4. `compile()` to turn it into something you can run with `.invoke()`.

Let's build the smallest possible graph, one that only chats.

In [ ]:
from langgraph.graph import StateGraph, START, END

mini = StateGraph(HelpdeskState)
mini.add_node("chat", chat)
mini.add_edge(START, "chat")
mini.add_edge("chat", END)
mini_app = mini.compile()

result = mini_app.invoke({"messages": [HumanMessage(content="Hi! Who are you?")]})
print(result["messages"][-1].text)

The graph returns the **final state**. The reply is the last message in `result["messages"]`.

You can print any graph as a **Mermaid** diagram. Paste the output into [mermaid.live](https://mermaid.live) to see it drawn.

In [ ]:
print(mini_app.get_graph().draw_mermaid())

## 8. Conditional routing

A normal edge always goes to the same node. A **conditional edge** chooses the next node while the graph runs, using a **routing function**:

- It receives the state.
- It returns the **name** of the next node.

Our routing function reads the `route` chosen by `classify` and maps it to a node name.

In [ ]:
def route_request(state: HelpdeskState) -> str:
    # Pick the next node based on the route chosen by classify().
    routes = {"question": "retrieve", "calculation": "calculator_agent", "action": "draft_action", "chat": "chat"}
    return routes.get(state["route"], "chat")


print(route_request({"route": "question"}))
print(route_request({"route": "action"}))
print(route_request({"route": "something unexpected"}))

Notice the safe default: if the route is ever something unexpected, we fall back to `chat` instead of crashing.

We connect it with `add_conditional_edges`:

```python
graph.add_conditional_edges("classify", route_request, ["retrieve", "calculator_agent", "draft_action", "chat"])
```

The list names every node the router might choose, so LangGraph can check them and draw the diagram.

## 9. Build a simple LangGraph agent

Let's build the first version of the helpdesk with **two paths**: questions go to the knowledge base, and everything else goes to chat.

The question path has two nodes:

- `retrieve` uses the **retriever** from Section 3 to find relevant documents.
- `answer` uses a **chain** (`ANSWER_PROMPT | llm`) to answer from those documents and list the sources.

In [ ]:
def retrieve(state: HelpdeskState) -> dict:
    # Search the knowledge base for the user's question.
    documents = retriever.invoke(last_user_message(state))
    context = "\n\n".join(f"[{d.metadata['title']}]\n{d.page_content}" for d in documents)
    sources = [d.metadata["title"] for d in documents]
    print(f"  [retrieve] found: {sources}")
    return {"context": context, "sources": sources}


def answer(state: HelpdeskState) -> dict:
    # Answer the question using the retrieved context (a LangChain chain: prompt | model).
    chain = ANSWER_PROMPT | llm
    reply = chain.invoke({"context": state["context"], "messages": conversation(state)})
    text = reply.text + "\n\nSources: " + ", ".join(state["sources"])
    return {"messages": [AIMessage(content=text)]}


print("retrieve and answer nodes ready.")

In [ ]:
def build_version_1():
    graph = StateGraph(HelpdeskState)

    graph.add_node("classify", classify)
    graph.add_node("retrieve", retrieve)
    graph.add_node("answer", answer)
    graph.add_node("chat", chat)

    graph.add_edge(START, "classify")
    graph.add_conditional_edges("classify", route_request, ["retrieve", "chat"])
    graph.add_edge("retrieve", "answer")
    graph.add_edge("answer", END)
    graph.add_edge("chat", END)

    return graph.compile()


app_v1 = build_version_1()

for question in ["What is prompt injection?", "Hi there, how's it going?"]:
    print(f"\nYou: {question}")
    result = app_v1.invoke({"messages": [HumanMessage(content=question)]})
    print(f"Agent: {result['messages'][-1].text}")

Watch the `[classify]` and `[retrieve]` lines. They show which path each request took.

Version 1 only knows two paths. If `classify` picks `calculation` or `action`, the routing function returns a node that doesn't exist yet, and the graph raises an error. We'll add those paths next.

## 10. Add a tool and route based on the user's request

Now the **calculation** path. It's a small agent loop inside the graph:

```text
calculator_agent ──(asked for a tool)──→ tools ──→ calculator_agent ──(final answer)──→ END
```

LangGraph provides two ready-made pieces for this:

| Piece | What it does |
|---|---|
| `ToolNode(TOOLS)` | A node that runs whatever tools the last AI message asked for, and adds the results as `ToolMessage`s |
| `tools_condition` | A routing function: goes to `"tools"` if the last message has tool calls, otherwise to `END` |

This is the same Reason → Act → Observe loop from Lesson 4, written as two nodes and two edges. LangGraph also stops runaway loops for you with a **recursion limit** (25 steps by default).

In [ ]:
def calculator_agent(state: HelpdeskState) -> dict:
    # An LLM with the calculator tool. It may ask for the tool, or give its final answer.
    model_with_tools = llm.bind_tools(TOOLS)
    reply = model_with_tools.invoke([SystemMessage(content=CALCULATOR_PROMPT)] + current_turn(state))
    if reply.tool_calls:
        print(f"  [calculator_agent] calls: {[(c['name'], c['args']) for c in reply.tool_calls]}")
    return {"messages": [reply]}


from langgraph.prebuilt import ToolNode, tools_condition


def build_version_2(checkpointer=None):
    graph = StateGraph(HelpdeskState)

    graph.add_node("classify", classify)
    graph.add_node("retrieve", retrieve)
    graph.add_node("answer", answer)
    graph.add_node("calculator_agent", calculator_agent)
    graph.add_node("tools", ToolNode(TOOLS))
    graph.add_node("chat", chat)

    graph.add_edge(START, "classify")
    graph.add_conditional_edges("classify", route_request, ["retrieve", "calculator_agent", "chat"])

    graph.add_edge("retrieve", "answer")
    graph.add_edge("answer", END)

    graph.add_conditional_edges("calculator_agent", tools_condition)   # tool call -> "tools", otherwise -> END
    graph.add_edge("tools", "calculator_agent")                         # results go back to the agent

    graph.add_edge("chat", END)

    return graph.compile(checkpointer=checkpointer)


app_v2 = build_version_2()

result = app_v2.invoke({"messages": [HumanMessage(content="What is 15% of 2,480, plus 99?")]})
print(f"Agent: {result['messages'][-1].text}\n")

print("Messages in this turn:")
for message in result["messages"]:
    print(f"  {type(message).__name__:12}", message.tool_calls if getattr(message, "tool_calls", None) else message.text)

The message list shows the whole loop: question → tool call → tool result → answer. The router sent the request down the right path, and the calculator agent decided to use the tool.

## 11. Add basic persistence (checkpointing)

Try a follow-up with version 2 as it is:

In [ ]:
app_v2.invoke({"messages": [HumanMessage(content="What is RAG?")]})
result = app_v2.invoke({"messages": [HumanMessage(content="Can you explain that in one sentence for a 10-year-old?")]})
print(result["messages"][-1].text)

The agent doesn't know what "that" means. Each `.invoke()` started from an empty state, just like the stateless model in Lesson 6.

A **checkpointer** fixes this. It saves the state after every step, grouped by a **thread id**: one id per conversation.

- `InMemorySaver` keeps checkpoints in memory, which is perfect for learning. Real apps use a database, such as SQLite or Postgres.
- You pass the thread id in a `config` dictionary every time you call `.invoke()`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

app_v2 = build_version_2(checkpointer=InMemorySaver())

thread_a = {"configurable": {"thread_id": "student-a"}}

for message in ["What is RAG?", "Can you explain that in one sentence for a 10-year-old?"]:
    print(f"\nYou: {message}")
    result = app_v2.invoke({"messages": [HumanMessage(content=message)]}, thread_a)
    print(f"Agent: {result['messages'][-1].text}")

Now the follow-up works, because the second call loaded the saved messages from thread `student-a`.

A **different thread** is a separate conversation, so it knows nothing about thread A:

In [ ]:
thread_b = {"configurable": {"thread_id": "student-b"}}
result = app_v2.invoke({"messages": [HumanMessage(content="What did I just ask you about?")]}, thread_b)
print("Thread B:", result["messages"][-1].text)

saved = app_v2.get_state(thread_a)
print("\nMessages saved in thread A:", len(saved.values["messages"]))

`get_state(config)` lets you look inside any saved conversation. Compare this with Lesson 6: the checkpointer gives you **short-term memory** (the conversation in a thread). Long-term facts about a user still need a separate store like your `memory.json`.

## 12. A simple human-in-the-loop step

Some actions should never happen without a person saying yes: sending messages, payments, deleting things. That's the **assisted** autonomy you learned about in Lesson 1.

LangGraph makes this easy with `interrupt()`:

1. A node calls `interrupt(...)` with information for the human.
2. The graph **pauses** and saves its state with the checkpointer. (That's why a checkpointer is required.)
3. Your program shows the request to a person and gets their answer.
4. You **resume** the graph with `Command(resume=answer)`. Inside the node, `interrupt()` returns that answer, and the node carries on.

Our action path has two nodes:

- `draft_action` turns the request into a message (structured output: who it's to, and the text). Nothing is sent.
- `human_approval` pauses for approval, then sends the message or cancels it.

Sending is simulated with `send_message()`, which saves to a pretend outbox, so nothing is actually sent anywhere.

> **Important:** when the graph resumes, the paused node **runs again from the beginning**. Never put anything with side effects (like sending) *before* `interrupt()`, or it would happen twice.

In [ ]:
SENT_MESSAGES = []   # a pretend outbox, so nothing is really sent


def send_message(to: str, text: str) -> str:
    """Pretend to send a message. In a real product this would call an email or chat API."""
    SENT_MESSAGES.append({"to": to, "text": text})
    return f"Message sent to {to}."


def draft_action(state: HelpdeskState) -> dict:
    # Turn the request into a message draft. Nothing is sent yet.
    drafter = llm.with_structured_output(MessageDraft)
    draft = drafter.invoke([SystemMessage(content=DRAFT_PROMPT), HumanMessage(content=last_user_message(state))])
    print(f"  [draft_action] to: {draft.to} | text: {draft.text}")
    return {"draft": draft.model_dump()}


from langgraph.types import interrupt, Command


def human_approval(state: HelpdeskState) -> dict:
    # HUMAN-IN-THE-LOOP: pause the graph and wait for a person to approve the draft.
    draft = state["draft"]
    decision = interrupt({"question": "Send this message? (yes/no)", "to": draft["to"], "text": draft["text"]})

    if str(decision).strip().lower() in ("yes", "y", "approve", "approved"):
        result = send_message(draft["to"], draft["text"])
    else:
        result = "Cancelled. Nothing was sent."
    print(f"  [human_approval] {result}")
    return {"action_result": result, "messages": [AIMessage(content=result)]}


print("Action nodes ready.")

Now the **final graph**, with all four paths.

In [ ]:
def build_graph(checkpointer=None):
    graph = StateGraph(HelpdeskState)

    graph.add_node("classify", classify)
    graph.add_node("retrieve", retrieve)
    graph.add_node("answer", answer)
    graph.add_node("calculator_agent", calculator_agent)
    graph.add_node("tools", ToolNode(TOOLS))
    graph.add_node("chat", chat)
    graph.add_node("draft_action", draft_action)
    graph.add_node("human_approval", human_approval)

    graph.add_edge(START, "classify")
    graph.add_conditional_edges("classify", route_request,
                                ["retrieve", "calculator_agent", "draft_action", "chat"])

    graph.add_edge("retrieve", "answer")
    graph.add_edge("answer", END)

    graph.add_conditional_edges("calculator_agent", tools_condition)
    graph.add_edge("tools", "calculator_agent")

    graph.add_edge("draft_action", "human_approval")
    graph.add_edge("human_approval", END)

    graph.add_edge("chat", END)

    return graph.compile(checkpointer=checkpointer or InMemorySaver())


app = build_graph()
print(app.get_graph().draw_mermaid())

Paste the diagram into [mermaid.live](https://mermaid.live). Dotted arrows are conditional edges; solid arrows are normal edges.

Let's ask the agent to send a message. Watch it stop **before** sending.

In [ ]:
config = {"configurable": {"thread_id": "approval-demo"}}

result = app.invoke(
    {"messages": [HumanMessage(content="Send a reminder to my study group that we meet at 6pm on Friday.")]},
    config,
)

print("\nPaused?", "__interrupt__" in result)
print("Waiting at node:", app.get_state(config).next)
print("Approval request:", result["__interrupt__"][0].value)
print("Outbox:", SENT_MESSAGES)

The graph is paused at `human_approval` and the outbox is still empty. Now a human approves:

In [ ]:
result = app.invoke(Command(resume="yes"), config)

print("Agent:", result["messages"][-1].text)
print("Outbox:", SENT_MESSAGES)

> **Try it:** run the two cells above again with a new `thread_id` and resume with `"no"`. The message should be cancelled, and the outbox shouldn't change.

## 13. Test the workflow

To test everything at once, we use a small helper, `run_turn()`. It sends a message, and if the graph pauses for approval, it shows the draft, gets a decision and resumes. In the tests the decision is set in advance; when you chat, it asks you.

Each state key is reset at the start of a turn (except `messages`, which keeps growing), so nothing from a previous turn leaks into the next one.

In [ ]:
def run_turn(app, message: str, thread_id: str, approve=None) -> dict:
    # Send one message in a conversation thread. Handles the approval pause if the graph stops for it.
    config = {"configurable": {"thread_id": thread_id}}
    fresh = {"messages": [HumanMessage(content=message)], "route": "", "context": "",
             "sources": [], "draft": {}, "action_result": ""}
    result = app.invoke(fresh, config)

    if "__interrupt__" in result:
        request = result["__interrupt__"][0].value
        print(f"\n  APPROVAL NEEDED -> to: {request['to']}\n                     text: {request['text']}")
        decision = approve if approve is not None else input("  Send this message? (yes/no): ")
        print(f"  Human says: {decision}")
        result = app.invoke(Command(resume=decision), config)

    return result


def summarise_turn(result: dict) -> dict:
    tools_used = [m.name for m in current_turn(result) if isinstance(m, ToolMessage)]
    return {"route": result.get("route", ""), "reply": result["messages"][-1].text,
            "sources": result.get("sources", []), "tools_used": tools_used,
            "action_result": result.get("action_result", "")}


print("Test helpers ready.")

In [ ]:
TESTS = [
    {"message": "What is prompt injection?", "expect_route": "question",
     "expect_source": "Agent safety and guardrails"},
    {"message": "What is 15% of 2,480, plus 99?", "expect_route": "calculation",
     "expect_tool": "calculate", "expect_in_reply": "471"},
    {"message": "Hi! What can you help me with?", "expect_route": "chat"},
    {"message": "Send a reminder to my study group that we meet at 6pm on Friday.", "expect_route": "action",
     "approve": "yes", "expect_action": "sent"},
    {"message": "Email my instructor that I'll miss tomorrow's class.", "expect_route": "action",
     "approve": "no", "expect_action": "cancelled"},
]


app = build_graph()
checks = []

for i, test in enumerate(TESTS, 1):
    print(f"\n[{i}] You: {test['message']}")
    turn = summarise_turn(run_turn(app, test["message"], thread_id=f"test-{i}", approve=test.get("approve")))
    print(f"    Agent: {turn['reply']}")

    checks.append((f"Routed to {test['expect_route']}", turn["route"] == test["expect_route"]))
    if "expect_source" in test:
        checks.append((f"Retrieved '{test['expect_source']}'", test["expect_source"] in turn["sources"]))
    if "expect_tool" in test:
        checks.append((f"Used the {test['expect_tool']} tool", test["expect_tool"] in turn["tools_used"]))
    if "expect_in_reply" in test:
        checks.append((f"Reply contains {test['expect_in_reply']}", test["expect_in_reply"] in turn["reply"].replace(",", "")))
    if "expect_action" in test:
        checks.append((f"Message {test['expect_action']} after '{test['approve']}'",
                       test["expect_action"] in turn["action_result"].lower()))

print("\n=== Checks ===")
for name, ok in checks:
    print(("PASS " if ok else "CHECK"), name)
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed.")

### What to look for

| Request | Expected path |
|---|---|
| "What is prompt injection?" | `classify` → `retrieve` → `answer`, with **Agent safety and guardrails** as a source |
| "What is 15% of 2,480, plus 99?" | `classify` → `calculator_agent` → `tools` → `calculator_agent`, answer **471** |
| "Hi! What can you help me with?" | `classify` → `chat` |
| "Send a reminder…" (approved) | `classify` → `draft_action` → `human_approval` → **sent** |
| "Email my instructor…" (rejected) | `classify` → `draft_action` → `human_approval` → **cancelled** |

The router is an LLM, so it can occasionally choose differently. If a check fails, read the `[classify]` line: its reason usually explains the choice.

### Try it: chat with your helpdesk

Run the cell below. When the agent wants to send a message, it will ask you to approve it. Type `quit` to stop.

In [ ]:
chat_thread = "my-chat"
print("Chat with the helpdesk. Type 'quit' to stop.")
while True:
    message = input("\nYou: ").strip()
    if not message:
        continue
    if message.lower() in ("quit", "exit"):
        break
    result = run_turn(app, message, chat_thread)
    print("Agent:", result["messages"][-1].text)

## 14. Challenge

### Part A: Add a new route

Add a **quiz** path: when the user says something like *"Quiz me on embeddings"*, the agent retrieves the relevant document and writes **one multiple-choice question** with four options, without revealing the answer.

1. Add `"quiz"` to the `Literal` in `RouteDecision`, and describe it in `ROUTER_PROMPT`. Re-run those cells.
2. Write a `quiz` node that uses the retriever and the LLM. A prompt template works well here.
3. Update `route_request`, add the node to `build_graph`, and connect it with edges.
4. Print the new Mermaid diagram, and test it with two quiz requests.

### Part B: Let the human edit the draft

Right now, the human can only say yes or no. Change `human_approval` so they can also **edit** the message:

- If the answer is `"yes"`, send the draft.
- If the answer starts with `"edit:"`, send the edited text instead. For example, `edit: We meet at 7pm, not 6pm.`
- Anything else cancels.

Test all three by resuming with `Command(resume=...)`.

### Stretch

`InMemorySaver` forgets everything when the notebook restarts. Install `langgraph-checkpoint-sqlite` and switch to `SqliteSaver`, so conversations are saved to a file and survive a restart.

In [ ]:
# Part A: your quiz node.

QUIZ_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "TODO: tell the model to write ONE multiple-choice question with options A-D, "
               "based only on this context, and not to reveal the answer.\n\nContext:\n{context}"),
    ("human", "{request}"),
])


def quiz(state: HelpdeskState) -> dict:
    # TODO: 1) retrieve documents for last_user_message(state)
    #       2) join their page_content into one context string
    #       3) run (QUIZ_PROMPT | llm) with the context and the request
    #       4) return {"messages": [AIMessage(content=...)]}
    raise NotImplementedError("TODO: complete the quiz node")


# Then: update RouteDecision, ROUTER_PROMPT, route_request and build_graph, and test it:
# print(run_turn(build_graph(), "Quiz me on embeddings", thread_id="quiz-1")["messages"][-1].text)

In [ ]:
# Part B: test your new human_approval node. (Change it in Section 12, then re-run build_graph.)

# app = build_graph()
# config = {"configurable": {"thread_id": "edit-demo"}}
# app.invoke({"messages": [HumanMessage(content="Remind my study group we meet at 6pm.")]}, config)
# result = app.invoke(Command(resume="edit: Reminder: we meet at 7pm, not 6pm."), config)
# print(result["messages"][-1].text)
# print(SENT_MESSAGES[-1])

## Recap

| Concept | Where you used it |
|---|---|
| **Model** | `ChatGoogleGenerativeAI` with `.invoke()` and `.bind_tools()` |
| **Prompt template** | `ANSWER_PROMPT`, joined to the model as a chain with the pipe operator |
| **Tool** | `@tool calculate`, with the schema built from the function |
| **Retriever** | `InMemoryVectorStore(...).as_retriever()` over the Lesson 5 documents |
| **Agent** | `create_agent`, a one-line tool-using agent |
| **State** | `HelpdeskState`, with `add_messages` adding to the conversation |
| **Nodes** | `classify`, `retrieve`, `answer`, `calculator_agent`, `chat`, `draft_action`, `human_approval` |
| **Edges** | `add_edge`, from `START` to `END` |
| **Conditional routing** | `route_request` and `tools_condition` |
| **Persistence** | `InMemorySaver` and thread ids |
| **Human-in-the-loop** | `interrupt()` to pause, `Command(resume=...)` to continue |

**Next:** complete the starter file `lesson_08_langgraph_agent.py`, run its tests, and submit your work. See `README.md` for the submission checklist.